# MWU Feature Analysis

Two-sided Mann-Whitney U tests with Benjamini-Hochberg FDR correction,
comparing feature distributions between WGD=0 and WGD=1 subgenomes.

**To switch between species (e.g. Brassica → Maize):**
edit only the `⚙️ CONFIGURATION` cell below. All other cells are unchanged.

**Intended run order:** Restart kernel → Run All Cells.

In [12]:
import os
import numpy as np
import pandas as pd
from scipy import stats
from statsmodels.stats.multitest import multipletests
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from adjustText import adjust_text

---
## Configuration
**This is the only cell you need to edit between species or analyses.**

- `OUTPUT_DIR` — where results CSVs and figures are saved
- `METADATA_COLS` — columns to exclude from testing (non-feature columns, excluding the label)
- `DATASETS` — list of `{"name": ..., "path": ...}` dicts, one per CSV file
- `feature_rename_dict` — maps raw column names to display names for plots/CSVs

`LABEL_COL` and `ALPHA` rarely need changing but are here for completeness.

---

In [13]:
# =============================================================================
# USER CONFIGURATION — edit this cell only
# =============================================================================

# Directory where all output CSVs and figures will be saved.
# Change this path when switching species or projects.
OUTPUT_DIR = '/blue/meixiazhao/laylaaschuster/brapaGDM/final_brapaMLpreprocess/mwu_raw_feature_analysis'

# Binary label column name (0 vs 1). Unlikely to need changing.
LABEL_COL = 'WGD'

# FDR significance threshold.
ALPHA = 0.05

# Columns to exclude from feature testing (non-feature metadata).
# These differ between species:
#   Brassica : ['gene_id', 'location', 'group']
#   Maize    : ['Maize', 'location', 'group']
METADATA_COLS = ['gene_id', 'location', 'group']

# Dataset registry: one dict per CSV file to analyse.
# 'name' is used for output filenames and plot titles.
# 'path' is the full path to the input CSV.
DATASETS = [
    {
        'name': 'lf_mf1_all',
        'path': '/blue/meixiazhao/laylaaschuster/brapaGDM/final_brapaMLpreprocess/final_featuresets/Brapa_lf_mf1_all_columns_final.csv',
    },
    {
        'name': 'lf_mf2_all',
        'path': '/blue/meixiazhao/laylaaschuster/brapaGDM/final_brapaMLpreprocess/final_featuresets/Brapa_lf_mf2_all_columns_final.csv',
    },
    {
        'name': 'mf1_mf2_all',
        'path': '/blue/meixiazhao/laylaaschuster/brapaGDM/final_brapaMLpreprocess/final_featuresets/Brapa_mf1_mf2_all_columns_final.csv',
    },
    {
        'name': 'lf_mf1_g1',
        'path': '/blue/meixiazhao/laylaaschuster/brapaGDM/final_brapaMLpreprocess/final_featuresets/Brapa_lf_mf1_groupI_all_columns_final.csv',
    },
    {
        'name': 'lf_mf2_g1',
        'path': '/blue/meixiazhao/laylaaschuster/brapaGDM/final_brapaMLpreprocess/final_featuresets/Brapa_lf_mf2_groupI_all_columns_final.csv',
    },
    {
        'name': 'mf1_mf2_g1',
        'path': '/blue/meixiazhao/laylaaschuster/brapaGDM/final_brapaMLpreprocess/final_featuresets/Brapa_mf1_mf2_groupI_all_columns_final.csv',
    },
]

# Feature display name mapping.
# Keys are raw column names from the CSV; values are the names used in
# output CSVs and figure labels. Features not listed here are left as-is.
# This dict is species-specific — update when switching species.
feature_rename_dict = {
    # Location
    'location':                 'Loc',
    # GC content
    'gene_cds_gc':              'GCd',
    'promoter_gc':              'GCp',
    # Expression
    'avg_expression':           'Exp',
    'tau':                      '\u03C4',    # Greek tau — tissue specificity index
    # Evolutionary pressure
    'Ka':                       'Ka',
    'Ks':                       'Ks',
    'omega':                    '\u03C9',    # Greek omega (Ka/Ks ratio)
    # ACR
    'upstream_distance':        'ACRdu',
    'downstream_distance':      'ACRdd',
    'summit_fold_enrichment':   'ACRs',
    # Recombination
    'rec_rate':                 'Re',
    # Methylation — average (9)
    'CHH_up_ave':               'M1',
    'CHH_down_ave':             'M2',
    'CHH_body_ave':             'M3',
    'CHG_up_ave':               'M4',
    'CHG_down_ave':             'M5',
    'CHG_body_ave':             'M6',
    'CG_up_ave':                'M7',
    'CG_down_ave':              'M8',
    'CG_body_ave':              'M9',
    # Methylation — max (6)
    'CHH_up_max':               'M10',
    'CHH_down_max':             'M11',
    'CHG_up_max':               'M12',
    'CHG_down_max':             'M13',
    'CG_up_max':                'M14',
    'CG_down_max':              'M15',
    # TE
    'te_dist':                  'TEdi',
    'TEdenseAvgUp':             'TEu1',
    'TEdenseAvgDown':           'TEd1',
    'TEdenseMaxUp':             'TEu2',
    'TEdenseMaxDown':           'TEd2',
    'TEGroup_1':                'TE1',      # RC/Helitron
    'TEGroup_2':                'TE2',      # DNA transposons
    'TEGroup_3':                'TE3',      # LTR retrotransposons
    'TEGroup_4':                'TE4',      # Non-LTR retrotransposons
    # Histone modifications
    'H3K27ac_down':             'Hd5',
    'H3K27me3_down':            'Hd6',
    'H3K4me3_down':             'Hd3',
    'H3K27ac_genebody':         'Hg5',
    'H3K27me3_genebody':        'Hg6',
    'H3K4me3_genebody':         'Hg3',
    'H3K27ac_up':               'Hu5',
    'H3K27me3_up':              'Hu6',
    'H3K4me3_up':               'Hu3',
}

print(f'Configuration loaded.')
print(f'  Output directory : {OUTPUT_DIR}')
print(f'  Label column     : {LABEL_COL}')
print(f'  Alpha threshold  : {ALPHA}')
print(f'  Metadata columns : {METADATA_COLS}')
print(f'  Datasets to run  : {[d["name"] for d in DATASETS]}')

Configuration loaded.
  Output directory : /blue/meixiazhao/laylaaschuster/brapaGDM/final_brapaMLpreprocess/mwu_raw_feature_analysis
  Label column     : WGD
  Alpha threshold  : 0.05
  Metadata columns : ['gene_id', 'location', 'group']
  Datasets to run  : ['lf_mf1_all', 'lf_mf2_all', 'mf1_mf2_all', 'lf_mf1_g1', 'lf_mf2_g1', 'mf1_mf2_g1']


---
## Function Definitions
**Do not edit below this line.**
These cells define `run_mwu_analysis`, `plot_volcano`, and `plot_lollipop`.
They must be run once per session before the analysis loop.

---

In [14]:
def run_mwu_analysis(
    csv_path: str,
    label_col: str = 'WGD',
    metadata_cols: list = None,
    alpha: float = 0.05,
    output_dir: str = '.',
    output_filename: str = None,
    feature_rename_dict: dict = None,
) -> pd.DataFrame:
    """
    Run two-sided Mann-Whitney U tests with BH FDR correction on all features
    in a CSV dataset, comparing groups defined by a binary label column.

    NaNs are dropped per-feature so each test uses all available data,
    consistent with how XGBoost handled NaNs during modelling.

    Effect size: rank-biserial correlation r_rb = 2*U/(n0*n1) - 1
        r_rb > 0  -> WGD=0 group tends to have higher values
        r_rb < 0  -> WGD=1 group tends to have higher values
    """

    # -------------------------------------------------------------------------
    # 1. Validate and prepare
    # -------------------------------------------------------------------------

    if metadata_cols is None:
        metadata_cols = ['Maize', 'location', 'group']

    cols_to_exclude = set(metadata_cols) | {label_col}

    # -------------------------------------------------------------------------
    # 2. Load data
    # -------------------------------------------------------------------------

    print(f'Loading: {csv_path}')
    df = pd.read_csv(csv_path)
    print(f'  Shape: {df.shape[0]} rows x {df.shape[1]} columns')

    if label_col not in df.columns:
        raise ValueError(f"Label column '{label_col}' not found in dataset.")

    label_values = df[label_col].dropna().unique()
    if not set(label_values).issubset({0, 1}):
        raise ValueError(
            f"Label column '{label_col}' must contain only 0 and 1. "
            f"Found: {label_values}"
        )

    feature_cols = [c for c in df.columns if c not in cols_to_exclude]
    print(f'  Features to test : {len(feature_cols)}')
    print(f'  Excluded columns : {sorted(cols_to_exclude)}')

    df_wgd0 = df[df[label_col] == 0]
    df_wgd1 = df[df[label_col] == 1]
    print(f'  {label_col}=0 samples : {len(df_wgd0)}')
    print(f'  {label_col}=1 samples : {len(df_wgd1)}')

    # -------------------------------------------------------------------------
    # 3. Run MWU test per feature
    # -------------------------------------------------------------------------

    results = []

    for feature in feature_cols:

        # Drop NaNs independently per group to preserve all available data
        vals_0 = df_wgd0[feature].dropna().values
        vals_1 = df_wgd1[feature].dropna().values

        if len(vals_0) == 0 or len(vals_1) == 0:
            print(f"  WARNING: '{feature}' has no valid data in one or both groups. Skipping.")
            continue

        # Two-sided MWU; U is for the WGD=0 group (first argument)
        u_stat, p_raw = stats.mannwhitneyu(vals_0, vals_1, alternative='two-sided')

        med_0    = np.nanmedian(vals_0)
        med_1    = np.nanmedian(vals_1)
        med_diff = med_1 - med_0          # positive = WGD=1 higher

        n0 = len(vals_0)
        n1 = len(vals_1)

        # Rank-biserial correlation: r > 0 means WGD=0 tends higher
        rank_biserial_r = (2 * u_stat) / (n0 * n1) - 1

        direction = 'WGD1_higher' if med_diff > 0 else 'WGD0_higher'

        results.append({
            'feature':         feature,
            'n_WGD0':          n0,
            'n_WGD1':          n1,
            'median_WGD0':     med_0,
            'median_WGD1':     med_1,
            'median_diff':     med_diff,
            'U_statistic':     u_stat,
            'p_value_raw':     p_raw,
            'rank_biserial_r': rank_biserial_r,
            'direction':       direction,
        })

    results_df = pd.DataFrame(results)

    # -------------------------------------------------------------------------
    # 4. Benjamini-Hochberg FDR correction across all features
    # -------------------------------------------------------------------------

    reject, p_corrected, _, _ = multipletests(
        results_df['p_value_raw'].values,
        alpha=alpha,
        method='fdr_bh'
    )

    results_df['p_value_BH']  = p_corrected
    results_df['significant'] = reject

    results_df = results_df[[
        'feature', 'n_WGD0', 'n_WGD1',
        'median_WGD0', 'median_WGD1', 'median_diff',
        'U_statistic', 'p_value_raw', 'p_value_BH',
        'rank_biserial_r', 'direction', 'significant',
    ]]

    results_df = results_df.sort_values('p_value_BH').reset_index(drop=True)

    # Apply display name renaming before saving
    if feature_rename_dict is not None:
        results_df['feature'] = results_df['feature'].map(
            lambda x: feature_rename_dict.get(x, x)
        )

    # -------------------------------------------------------------------------
    # 5. Summary and save
    # -------------------------------------------------------------------------

    n_sig = results_df['significant'].sum()
    print(f'  Results summary:')
    print(f'    Features tested            : {len(results_df)}')
    print(f'    Significant (BH p < {alpha}) : {n_sig}')
    print(f'    Not significant            : {len(results_df) - n_sig}')

    if output_filename is None:
        base = os.path.splitext(os.path.basename(csv_path))[0]
        output_filename = f'{base}_mwu_results.csv'

    output_path = os.path.join(output_dir, output_filename)
    results_df.to_csv(output_path, index=False)
    print(f'  Saved: {output_path}\n')

    return results_df

In [15]:
# =============================================================================
# Shared colour palettes
# Defined here so volcano and lollipop functions stay visually consistent.
# =============================================================================

# Volcano: 3-category scheme, avoids red/green
VOLCANO_COLORS = {
    'WGD0_higher_sig': '#2166AC',   # blue  — significant, WGD=0 higher
    'WGD1_higher_sig': '#E08214',   # amber — significant, WGD=1 higher
    'not_significant': '#AAAAAA',   # grey  — not significant
}

# Lollipop: 4-category scheme (light/dark x WGD0/WGD1)
LOLLIPOP_COLORS = {
    'WGD0_sig':    '#0571B0',   # dark blue
    'WGD0_nonsig': '#92C5DE',   # light blue
    'WGD1_sig':    '#008837',   # dark green
    'WGD1_nonsig': '#A6DBA0',   # light green
}

In [16]:
def plot_volcano(
    results_df: pd.DataFrame,
    dataset_name: str,
    alpha: float = 0.05,
    effect_size_threshold: float = 0.1,
    output_dir: str = '.',
    feature_rename_dict: dict = None,
) -> None:
    """
    Volcano plot: x = rank-biserial r, y = -log10(BH p-value).
    Only significant features are labelled. Saves 600 DPI PNG.
    """

    # Layout parameters — adjust here if needed
    FIGURE_WIDTH_IN  = 3.5
    FIGURE_HEIGHT_IN = 4.0
    POINT_SIZE       = 18
    POINT_ALPHA      = 0.85
    LABEL_FONTSIZE   = 4
    AXIS_FONTSIZE    = 8
    TITLE_FONTSIZE   = 9
    LINE_WIDTH       = 0.8
    LINE_STYLE       = '--'

    # --- Prepare data ---
    df = results_df.copy()
    
    if feature_rename_dict is not None:
        df['feature'] = df['feature'].map(lambda x: feature_rename_dict.get(x, x))

    p_floor = 1e-300
    df['neg_log10_p'] = -np.log10(df['p_value_BH'].clip(lower=p_floor))

    def assign_volcano_color(row):
        if not row['significant']:
            return 'not_significant'
        return 'WGD0_higher_sig' if row['direction'] == 'WGD0_higher' else 'WGD1_higher_sig'

    df['color_category'] = df.apply(assign_volcano_color, axis=1)
    df['color'] = df['color_category'].map(VOLCANO_COLORS)
    y_threshold = -np.log10(alpha)

    # --- Build figure ---
    fig, ax = plt.subplots(figsize=(FIGURE_WIDTH_IN, FIGURE_HEIGHT_IN))

    # Plot non-significant first (behind significant)
    for category in ['not_significant', 'WGD0_higher_sig', 'WGD1_higher_sig']:
        subset = df[df['color_category'] == category]
        ax.scatter(
            subset['rank_biserial_r'], subset['neg_log10_p'],
            color=VOLCANO_COLORS[category], s=POINT_SIZE, alpha=POINT_ALPHA,
            linewidths=0, zorder=2 if category == 'not_significant' else 3,
        )

    # Reference lines
    ax.axhline(y=y_threshold, color='black', linestyle=LINE_STYLE,
               linewidth=LINE_WIDTH, zorder=1)
    if effect_size_threshold > 0:
        for x_val in [-effect_size_threshold, effect_size_threshold]:
            ax.axvline(x=x_val, color='black', linestyle=LINE_STYLE,
                       linewidth=LINE_WIDTH, zorder=1)

    # Label significant features with adjustText
    sig_df = df[df['significant']]
    texts = []
    for _, row in sig_df.iterrows():
        t = ax.text(
            row['rank_biserial_r'], row['neg_log10_p'], row['feature'],
            fontsize=LABEL_FONTSIZE, ha='center', va='bottom', color='black',
        )
        texts.append(t)

    if texts:
        adjust_text(texts, ax=ax,
                    arrowprops=dict(arrowstyle='-', color='black', lw=0.4),
                    expand=(1.2, 1.4))

    # Axes formatting
    ax.set_xlabel('Rank-biserial correlation ($r_{rb}$)', fontsize=AXIS_FONTSIZE)
    ax.set_ylabel('$-\\log_{10}$(BH-adjusted $p$-value)', fontsize=AXIS_FONTSIZE)
    ax.set_title(dataset_name, fontsize=TITLE_FONTSIZE, pad=6)
    ax.tick_params(labelsize=AXIS_FONTSIZE)

    x_abs_max = max(abs(df['rank_biserial_r'].min()), abs(df['rank_biserial_r'].max()))
    x_lim = min(x_abs_max * 1.15, 1.0)
    ax.set_xlim(-x_lim, x_lim)
    ax.axvline(x=0, color='black', linewidth=0.4, linestyle='-', zorder=1)

    # # Legend
    # legend_handles = [
    #     mpatches.Patch(color=VOLCANO_COLORS['WGD0_higher_sig'],
    #                    label='Sig., WGD=0 higher ($r_{rb}$ > 0)'),
    #     mpatches.Patch(color=VOLCANO_COLORS['WGD1_higher_sig'],
    #                    label='Sig., WGD=1 higher ($r_{rb}$ < 0)'),
    #     mpatches.Patch(color=VOLCANO_COLORS['not_significant'],
    #                    label='Not significant'),
    # ]
    # ax.legend(
    #     handles=legend_handles, 
    #     fontsize=6, 
    #     frameon=False, 
    #     bbox_to_anchor=(1.02, 1.0),
    #     loc='upper left'
    # )

    # Save
    plt.tight_layout()
    output_path = os.path.join(output_dir, f'{dataset_name}_volcano.png')
    fig.savefig(output_path, dpi=600, bbox_inches='tight')
    plt.close(fig)
    print(f'  Volcano saved : {output_path}')

In [17]:
def plot_lollipop(
    results_df: pd.DataFrame,
    dataset_name: str,
    output_dir: str = '.',
    feature_rename_dict: dict = None,
) -> None:
    """
    Ranked lollipop plot: features sorted by |r_rb|, largest at top.
    Colour encodes direction x significance. Saves 600 DPI PNG.
    """

    # Layout parameters — adjust here if needed
    FIGURE_WIDTH_IN    = 3.5
    HEIGHT_PER_FEATURE = 0.25   # dynamic height scaling
    HEIGHT_MIN_IN      = 3.0
    DOT_SIZE           = 50
    STEM_LINEWIDTH     = 1.2
    LABEL_FONTSIZE     = 7
    AXIS_FONTSIZE      = 8
    TITLE_FONTSIZE     = 9
    ZERO_LINE_WIDTH    = 0.8

    # --- Prepare data ---
    df = results_df.copy()
    
    if feature_rename_dict is not None:
        df['feature'] = df['feature'].map(lambda x: feature_rename_dict.get(x, x))

    def assign_lollipop_color(row):
        if row['direction'] == 'WGD0_higher':
            return 'WGD0_sig' if row['significant'] else 'WGD0_nonsig'
        return 'WGD1_sig' if row['significant'] else 'WGD1_nonsig'

    df['color_category'] = df.apply(assign_lollipop_color, axis=1)
    df['color'] = df['color_category'].map(LOLLIPOP_COLORS)

    # Sort by |r| ascending so largest appears at the top of the plot
    df['abs_r'] = df['rank_biserial_r'].abs()
    df = df.sort_values('abs_r', ascending=True).reset_index(drop=True)

    n_features  = len(df)
    y_positions = np.arange(n_features)

    # --- Build figure with dynamic height ---
    fig_height = max(HEIGHT_MIN_IN, n_features * HEIGHT_PER_FEATURE)
    fig, ax = plt.subplots(figsize=(FIGURE_WIDTH_IN, fig_height))

    # Draw lollipops
    for i, (_, row) in enumerate(df.iterrows()):
        color = row['color']
        r_val = row['rank_biserial_r']
        ax.hlines(y=i, xmin=0, xmax=r_val, colors=color,
                  linewidth=STEM_LINEWIDTH, zorder=2)
        ax.scatter(x=r_val, y=i, color=color, s=DOT_SIZE,
                   zorder=3, linewidths=0)

    # Reference line at x=0
    ax.axvline(x=0, color='black', linewidth=ZERO_LINE_WIDTH,
               linestyle='-', zorder=1)

    # Axes formatting
    ax.set_yticks(y_positions)
    ax.set_yticklabels(df['feature'], fontsize=LABEL_FONTSIZE)
    ax.set_xlabel('Rank-biserial correlation ($r_{rb}$)', fontsize=AXIS_FONTSIZE)
    ax.set_title(dataset_name, fontsize=TITLE_FONTSIZE, pad=6)
    ax.tick_params(axis='x', labelsize=AXIS_FONTSIZE)

    x_abs_max = df['rank_biserial_r'].abs().max()
    x_lim = min(x_abs_max * 1.2, 1.0)
    ax.set_xlim(-x_lim, x_lim)

    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)

    # # Legend
    # legend_handles = [
    #     mpatches.Patch(color=LOLLIPOP_COLORS['WGD0_sig'],    label='WGD=0 higher, sig.'),
    #     mpatches.Patch(color=LOLLIPOP_COLORS['WGD0_nonsig'], label='WGD=0 higher, n.s.'),
    #     mpatches.Patch(color=LOLLIPOP_COLORS['WGD1_sig'],    label='WGD=1 higher, sig.'),
    #     mpatches.Patch(color=LOLLIPOP_COLORS['WGD1_nonsig'], label='WGD=1 higher, n.s.'),
    # ]
    # ax.legend(handles=legend_handles, fontsize=6, frameon=False, loc='lower right')

    # Save
    plt.tight_layout()
    output_path = os.path.join(output_dir, f'{dataset_name}_lollipop.png')
    fig.savefig(output_path, dpi=600, bbox_inches='tight')
    plt.close(fig)
    print(f'  Lollipop saved : {output_path}')

---
## Run Analysis
Runs MWU analysis, volcano plot, and lollipop plot for every dataset
defined in `DATASETS`. Results are stored in `all_results` (dict keyed
by dataset name) and saved to `OUTPUT_DIR`.

---

In [18]:
# Dictionary to store results DataFrames, keyed by dataset name.
# Access individual results as: all_results['lf_mf1_all']
all_results = {}

for dataset in DATASETS:

    name = dataset['name']
    path = dataset['path']

    print(f'=' * 60)
    print(f'Dataset: {name}')
    print(f'=' * 60)

    # --- Run MWU analysis and save results CSV ---
    results = run_mwu_analysis(
        csv_path=path,
        label_col=LABEL_COL,
        metadata_cols=METADATA_COLS,
        alpha=ALPHA,
        output_dir=OUTPUT_DIR,
        feature_rename_dict=feature_rename_dict,
    )

    # --- Volcano plot (main figure) ---
    plot_volcano(
        results_df=results,
        dataset_name=name,
        alpha=ALPHA,
        effect_size_threshold=0.1,
        output_dir=OUTPUT_DIR,
        feature_rename_dict=feature_rename_dict,
    )

    # --- Lollipop plot (supplemental figure) ---
    plot_lollipop(
        results_df=results,
        dataset_name=name,
        output_dir=OUTPUT_DIR,
        feature_rename_dict=feature_rename_dict,
    )

    all_results[name] = results

print()
print('All datasets complete.')
print(f'Results stored in: all_results')
print(f'Keys: {list(all_results.keys())}')

Dataset: lf_mf1_all
Loading: /blue/meixiazhao/laylaaschuster/brapaGDM/final_brapaMLpreprocess/final_featuresets/Brapa_lf_mf1_all_columns_final.csv
  Shape: 4724 rows x 48 columns
  Features to test : 44
  Excluded columns : ['WGD', 'gene_id', 'group', 'location']
  WGD=0 samples : 2362
  WGD=1 samples : 2362
  Results summary:
    Features tested            : 44
    Significant (BH p < 0.05) : 16
    Not significant            : 28
  Saved: /blue/meixiazhao/laylaaschuster/brapaGDM/final_brapaMLpreprocess/mwu_raw_feature_analysis/Brapa_lf_mf1_all_columns_final_mwu_results.csv

  Volcano saved : /blue/meixiazhao/laylaaschuster/brapaGDM/final_brapaMLpreprocess/mwu_raw_feature_analysis/lf_mf1_all_volcano.png
  Lollipop saved : /blue/meixiazhao/laylaaschuster/brapaGDM/final_brapaMLpreprocess/mwu_raw_feature_analysis/lf_mf1_all_lollipop.png
Dataset: lf_mf2_all
Loading: /blue/meixiazhao/laylaaschuster/brapaGDM/final_brapaMLpreprocess/final_featuresets/Brapa_lf_mf2_all_columns_final.csv
  Shap